### Fraud Detection using Behavioural Risk Modelling & Machine Learning

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Sklearn - data split
from sklearn.model_selection import train_test_split

# Sklearn - preprocessing
from sklearn.preprocessing import StandardScaler

# Sklearn - models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest

# Sklearn - evaluation
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)

# Sklearn - calibration
from sklearn.calibration import calibration_curve

In [4]:
df2 = pd.read_csv("data/fraud_feature_engineered.csv")
df2.head(1)

,Transaction_ID,Customer_ID,Transaction_Date,Amount,Merchant_Category,Merchant_ID,Card_Type,Transaction_Type,Country,Is_International,...,Customer_Avg_Amount,Amount_vs_Avg,Above_Customer_Avg,No_Chip_No_Pin,High_Distance_And_Amount,International_And_Far,Night_No_Pin,High_Risk_Category,Device_Fraud_Rate,TxnType_Fraud_Rate
0,1,25795,2025-05-28 11:54:36,81.53,Online Services,8459,Gold,POS,Germany,1,...,190.42,0.428159,0,0,0,0,0,1,0.014424,0.015004


In [5]:
features = [
    # Core numeric
    "Amount_log",
    "Hour_of_Day",
    "Distance_From_Home",
    
    # Binary / behavior
    "Is_International",
    "Is_Chip",
    "Is_Pin_Used",
    
    # Engineered risk flags
    "High_Amount_Flag",
    "Is_Night",
    "Is_Weekend",
    "Far_From_Home",
    
    # Customer behavior
    "Customer_Avg_Amount",
    "Amount_vs_Avg",
    "Above_Customer_Avg",
    
    # Security / anomaly
    "No_Chip_No_Pin",
    
    # Interaction features
    "High_Distance_And_Amount",
    "International_And_Far",
    "Night_No_Pin",
    
    # Other Features - appeared weak previously as all ranges similarly
    "High_Risk_Category",
    "Device_Fraud_Rate",
    "TxnType_Fraud_Rate"
]

In [6]:
X = df2[features]
y = df2["Fraud_Flag"]

In [7]:
# Train/Test Split:

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)

In [8]:
X_train.isna().sum()

Amount_log                  0
Hour_of_Day                 0
Distance_From_Home          0
Is_International            0
Is_Chip                     0
Is_Pin_Used                 0
High_Amount_Flag            0
Is_Night                    0
Is_Weekend                  0
Far_From_Home               0
Customer_Avg_Amount         0
Amount_vs_Avg               0
Above_Customer_Avg          0
No_Chip_No_Pin              0
High_Distance_And_Amount    0
International_And_Far       0
Night_No_Pin                0
High_Risk_Category          0
Device_Fraud_Rate           0
TxnType_Fraud_Rate          0
dtype: int64

In [9]:
print("Train fraud rate:", y_train.mean())
print("Test fraud rate:", y_test.mean())

Train fraud rate: 0.015
Test fraud rate: 0.015


In [10]:
# Standardizing Features:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
# 1: Rule-based Scoring:

df2["Rule_Score"] = (
    0.25 * df2["High_Amount_Flag"] +
    0.20 * df2["Far_From_Home"] +
    0.20 * df2["Above_Customer_Avg"] +
    0.15 * df2["Is_Night"] +
    0.15 * df2["No_Chip_No_Pin"] +
    0.05 * df2["High_Distance_And_Amount"]
)

df2["Rule_Pred"] = (df2["Rule_Score"] >= 0.2).astype(int)

In [22]:
rule_score_test = df2.loc[X_test.index, "Rule_Score"]
rule_pred_test = df2.loc[X_test.index, "Rule_Pred"]

In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("*" * 50)
print("Rule-Based Model")
print("*" * 50)
print("Precision:", precision_score(y_test, rule_pred_test))
print("Recall   :", recall_score(y_test, rule_pred_test))
print("F1 Score :", f1_score(y_test, rule_pred_test))
print("ROC-AUC:", roc_auc_score(y_test, rule_score_test))
print("Brier Score:", brier_score_loss(y_test, rule_score_test))

**************************************************
Rule-Based Model
**************************************************
Precision: 0.014478103498875012
Recall   : 0.296
F1 Score : 0.02760593154475083
ROC-AUC: 0.49413392893401015
Brier Score: 0.06341297500000001


In [24]:
cm = confusion_matrix(y_test, rule_pred_test)
cm

array([[68277, 30223],
       [ 1056,   444]])

In [25]:
rule_score_test.describe()

count    100000.000000
mean          0.172833
std           0.153939
min           0.000000
25%           0.000000
50%           0.150000
75%           0.300000
max           1.000000
Name: Rule_Score, dtype: float64

In [73]:
# Logistic Regression 

log_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

log_model.fit(X_train_scaled, y_train)
log_probs = log_model.predict_proba(X_test_scaled)[:, 1]
log_pred = (log_probs >= 0.5).astype(int)

In [74]:
print("*" * 50)
print("Logistic Regression Model")
print("*" * 50)
print("Precision:", precision_score(y_test, log_pred))
print("Recall   :", recall_score(y_test, log_pred))
print("F1 Score :", f1_score(y_test, log_pred))
print("ROC-AUC:", roc_auc_score(y_test, log_probs))
print("Brier Score:", brier_score_loss(y_test, log_probs))

**************************************************
Logistic Regression Model
**************************************************
Precision: 0.015129760195443927
Recall   : 0.4706666666666667
F1 Score : 0.02931711064510101
ROC-AUC: 0.5051349441624365
Brier Score: 0.24984152185304614


In [75]:
log_cm = confusion_matrix(y_test, log_pred)
log_cm

array([[52543, 45957],
       [  794,   706]])

In [76]:
print(classification_report(y_test, log_pred))

              precision    recall  f1-score   support

           0       0.99      0.53      0.69     98500
           1       0.02      0.47      0.03      1500

    accuracy                           0.53    100000
   macro avg       0.50      0.50      0.36    100000
weighted avg       0.97      0.53      0.68    100000



In [78]:
# Isolation Forest:

iso_model = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)

In [ ]:
#Train only on NO Fault:
X_train_NOfault = X_train_scaled[y_train == 0]

iso_model.fit(X_train_NOfault)


,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.02
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [81]:
iso_scores = -iso_model.decision_function(X_test_scaled)

In [90]:
threshold = np.quantile(iso_scores, 0.90)
iso_pred = (iso_scores >= threshold).astype(int)

In [91]:
print("Precision:", precision_score(y_test, iso_pred))
print("Recall   :", recall_score(y_test, iso_pred))
print("F1 Score :", f1_score(y_test, iso_pred))

# normalizing scores to 0–1
iso_probs = (iso_scores - iso_scores.min()) / (iso_scores.max() - iso_scores.min() + 1e-9)

print("ROC-AUC:", roc_auc_score(y_test, iso_probs))
print("Brier Score:", brier_score_loss(y_test, iso_probs))

Precision: 0.0143
Recall   : 0.09533333333333334
F1 Score : 0.024869565217391306
ROC-AUC: 0.4926495600676819
Brier Score: 0.11163794153441256
